In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)
logging.getLogger("absl").setLevel(logging.ERROR)
logger = logging.getLogger(__name__)

In [ ]:
import dataclasses
import pathlib
import warnings
from collections.abc import Sequence
from typing import Literal

import dapper
import jax
import numpy as np
import pandas as pd
from geometry.surface import rz_fourier_desc
from util import pytree

from constellaration_update import types as constellaration_update_types
from constellaration_update.checkpoint import flax_nnx as flax_nnx_checkpoint
from constellaration_update.coilset import utils as coilset_utils
from constellaration_update.machine_learning import (
    model as model_definition,
)
from constellaration_update.machine_learning import (
    train,
    types,
)
from constellaration_update.metrics import metrics as metrics_utils
from constellaration_update.utils.types import runtime_check_array_sizes

In [ ]:
OUTPUTS_DIR = pathlib.Path("/home/devuser/tmp/outputs/constellaration_update/")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
@dataclasses.dataclass(frozen=True)
class PinnedRun:
    label: str
    checkpoint_ids: tuple[str, ...]
    model_type: Literal["mlp", "attention"]


def read_checkpoint(
    model_type: Literal["mlp", "attention"], checkpoint_id: str
) -> types.CoilPredictorCheckpoint | types.AttentionCoilPredictorCheckpoint:
    """Read the checkpoint stored at `checkpoint_id` with its declared type."""
    match model_type:
        case "mlp":
            return dapper.read(types.CoilPredictorCheckpoint, checkpoint_id)
        case "attention":
            return dapper.read(types.AttentionCoilPredictorCheckpoint, checkpoint_id)


def read_model_from_checkpoint(
    checkpoint: types.CoilPredictorCheckpoint | types.AttentionCoilPredictorCheckpoint,
) -> model_definition.CoilPredictor | model_definition.AttentionCoilPredictor:
    if isinstance(checkpoint.config, types.CoilPredictorConfig):
        model = flax_nnx_checkpoint.from_checkpoint(
            checkpoint, model_definition.CoilPredictor
        )
    elif isinstance(checkpoint.config, types.AttentionCoilPredictorConfig):
        model = flax_nnx_checkpoint.from_checkpoint(
            checkpoint, model_definition.AttentionCoilPredictor
        )
    else:
        raise TypeError(
            f"Unsupported checkpoint config type: {type(checkpoint.config).__name__}"
        )
    return model


PINNED_RUNS = [
    PinnedRun(
        label="mlp-v1",
        checkpoint_ids=("D8vydRWnyqvYzYM5kKX8aCL", "DUFzgxPiiyccj4MWGZJ4Yu6"),
        model_type="mlp",
    ),
    PinnedRun(
        label="attention-v1",
        checkpoint_ids=("D5UkknkdjhxbMvRajnkeVz8",),
        model_type="attention",
    ),
]

In [ ]:
@runtime_check_array_sizes
def predict_coilsets(
    model: model_definition.CoilPredictor | model_definition.AttentionCoilPredictor,
    eval_dataset: list[types.EvalData],
) -> list[types.EvalData]:
    """Restore the model and run it on every eval boundary in one batched call."""
    n_modes_coils_max = model.config.n_modes_coils_max
    n_max_fourier_order = (n_modes_coils_max - 1) // 2
    batched_call = jax.jit(
        jax.vmap(jax.tree_util.Partial(model, fourier_order=n_max_fourier_order))
    )
    boundaries = [eval_data.boundary for eval_data in eval_dataset]
    batched_boundaries = pytree.tree_stack(boundaries)
    requirement_metrics = [eval_data.requirement_metrics for eval_data in eval_dataset]
    batched_requirement_metrics = pytree.tree_stack(requirement_metrics)
    predicted_batched = batched_call(batched_boundaries, batched_requirement_metrics)
    predicted_list = pytree.tree_unstack(predicted_batched)

    updated_eval_dataset = []
    for eval_data, predicted in zip(eval_dataset, predicted_list, strict=True):
        updated_eval_data = eval_data.model_copy(
            update=dict(predicted_coilset=predicted)
        )
        updated_eval_dataset.append(updated_eval_data)
    return updated_eval_dataset

In [ ]:
AGGREGATE_SUFFIXES = ("/min", "/mean", "/max", "/std")

METRICS_SCALAR_FIELDS = (
    "n_coils_per_half_period",
    "on_axis_average_magnetic_field",
    "minor_radius",
    "linking_current_consistency",
    "toroidal_flux",
)

METRICS_ARRAY_FIELDS = (
    "normalized_field_error",
    "local_quadratic_flux",
    "quadratic_flux",
    "coil_to_coil_min_distances",
    "coil_to_plasma_min_distances",
    "coil_lengths",
    "coil_curvatures",
    "coil_current_lengths",
    "coil_integrated_curvatures",
    "coil_torsions",
    "coil_arclength_variances",
    "coil_currents",
    "coil_linking_numbers",
)

NORMALIZED_PROP_SPECS = (
    ("coil_to_coil_min_distances", "normalized_coil_to_coil_min_distances", False),
    ("coil_to_plasma_min_distances", "normalized_coil_to_plasma_min_distances", False),
    ("coil_lengths", "normalized_coil_lengths", False),
    ("coil_curvatures", "normalized_coil_curvatures", True),
    ("coil_torsions", "normalized_coil_torsions", True),
)


def flatten_stats(
    arr: list | jax.Array | float | int,
) -> tuple[float, float, float, float]:
    if isinstance(arr, (int, float)):
        return float(arr), float(arr), float(arr), 0.0
    flat = np.asarray(arr, dtype=float).ravel()
    if flat.size == 0:
        return 0.0, 0.0, 0.0, 0.0
    return (
        float(flat.min()),
        float(flat.mean()),
        float(flat.max()),
        float(flat.std()),
    )


def parse_metrics_record(
    metrics: constellaration_update_types.ConStellarationUpdateMetrics,
) -> dict:
    aggregates: dict = {}
    for field in METRICS_SCALAR_FIELDS:
        value = getattr(metrics, field)
        if value is None:
            print(field, value)
        aggregates[field] = float(getattr(metrics, field))

    for field in METRICS_ARRAY_FIELDS:
        value = getattr(metrics, field)
        if value is None:
            continue
        if isinstance(value, Sequence | jax.Array):
            mn, mean, mx, std = flatten_stats(value)
            aggregates[f"{field}/min"] = mn
            aggregates[f"{field}/mean"] = mean
            aggregates[f"{field}/max"] = mx
            aggregates[f"{field}/std"] = std
        else:
            aggregates[field] = float(value)

    minor_radius = float(metrics.minor_radius)
    for raw_field, norm_field, multiply in NORMALIZED_PROP_SPECS:
        value = getattr(metrics, raw_field)
        if value is None or not isinstance(value, Sequence | jax.Array):
            continue
        mn, mean, mx, std = flatten_stats(value)
        factor = minor_radius if multiply else 1.0 / minor_radius
        aggregates[f"{norm_field}/min"] = mn * factor
        aggregates[f"{norm_field}/mean"] = mean * factor
        aggregates[f"{norm_field}/max"] = mx * factor
        aggregates[f"{norm_field}/std"] = std * factor

    return aggregates

In [ ]:
def compute_metrics_errors(
    predictions: list[types.EvalData],
) -> pd.DataFrame:
    metrics_error_aggregates_list = []
    for i, eval_data in enumerate(predictions):
        with warnings.catch_warnings(action="ignore"):
            assert eval_data.predicted_coilset is not None
            predicted_metrics = metrics_utils.evaluate_coilset_metrics_from_boundary(
                boundary=rz_fourier_desc.to_desc_fourier_rz_toroidal_surface(
                    eval_data.boundary
                ),
                coilset=coilset_utils.constellaration_update_to_desc(
                    eval_data.predicted_coilset
                ),
            )
            true_metrics = dapper.read(
                constellaration_update_types.ConStellarationUpdateMetrics,
                eval_data.true_metrics,
            )
            true_metrics = true_metrics.model_copy(
                update=dict(
                    n_coils_per_half_period=eval_data.true_coilset.n_unique_coils
                )
            )
            metrics_error = jax.tree_util.tree_map(
                lambda x, y: None if x is None else (x - y),
                predicted_metrics,
                true_metrics,
                is_leaf=lambda x: x is None,
            )
            metrics_error.surf_eval_coords = true_metrics.surf_eval_coords
            metrics_error.coil_eval_params -= true_metrics.coil_eval_params
            metrics_error.n_coils_per_half_period = true_metrics.n_coils_per_half_period
            metrics_error.minor_radius = true_metrics.minor_radius
            metrics_error_aggregates = parse_metrics_record(metrics_error)
            metrics_error_aggregates["boundary_id"] = eval_data.boundary_id
            metrics_error_aggregates_list.append(metrics_error_aggregates)

        if i % 16 == 0:
            jax.clear_caches()
            logger.info(f"Processed {i} / {len(predictions)}")  # noqa: G004
    logger.info(f"Finished processing all {len(predictions)} samples.")  # noqa: G004
    return pd.DataFrame(metrics_error_aggregates_list)

In [ ]:
EVAL_SETTINGS = types.EvalSettings(
    n_eval=32,
)

PLOT_SETTINGS = types.PlotSettings(
    poincare_settings=(
        constellaration_update_types.ConStellarationUpdatePoincarePlotSettings()
    ),
)

COMPARE_SETTINGS = types.CompareSettings()

In [ ]:
eval_dataframe = train.load_dataframes(
    "train", relative_min_coil_to_plasma_distance_error_threshold=-1e10
)

In [ ]:
eval_dataset = train.load_dataset(
    "eval",
    relative_min_coil_to_plasma_distance_error_threshold=EVAL_SETTINGS.relative_min_coil_to_plasma_distance_error_threshold,
    n=EVAL_SETTINGS.n_eval,
)

In [ ]:
indices = jax.random.permutation(jax.random.PRNGKey(0), len(eval_dataset))
mock_predictions = [
    d.model_copy(update=dict(predicted_coilset=d.true_coilset))
    for d in [eval_dataset[i] for i in indices[:8]]
]
eval_dataset_errors_df = compute_metrics_errors(mock_predictions)
eval_dataset_errors_df.to_csv(
    OUTPUTS_DIR / "metrics_errors_validation.csv", index=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
for column in [
    "normalized_field_error/mean",
    "local_quadratic_flux/mean",
    "normalized_coil_to_coil_min_distances/mean",
    "normalized_coil_to_plasma_min_distances/mean",
    "coil_to_plasma_min_distances/mean",
]:
    fig, ax = plt.subplots()
    g = sns.histplot(eval_dataset_errors_df[column], ax=ax)
    plt.show()

In [ ]:
metrics_error_dfs = []
for pin in PINNED_RUNS:
    for checkpoint_id in pin.checkpoint_ids:
        logger.info("Evaluating pinned run %r seed=%s", pin.label, checkpoint_id)
        checkpoint = read_checkpoint(pin.model_type, checkpoint_id)
        model = read_model_from_checkpoint(checkpoint)
        logger.info(
            "Running predictions for pinned run %r seed=%s", pin.label, checkpoint_id
        )
        predictions = predict_coilsets(model=model, eval_dataset=eval_dataset)
        logger.info(
            "Computing metrics errors for pinned run %r seed=%s",
            pin.label,
            checkpoint_id,
        )
        metrics_errors_df = compute_metrics_errors(predictions)
        metrics_errors_df["checkpoint_id"] = checkpoint_id
        metrics_errors_df["model_type"] = pin.model_type
        metrics_errors_df["label"] = pin.label
        metrics_error_dfs.append(metrics_errors_df)
metrics_errors_df = pd.concat(metrics_error_dfs, ignore_index=True)
metrics_errors_df.to_parquet(OUTPUTS_DIR / "metrics_errors_df.parquet")

In [ ]:
fig, ax = plt.subplots()
ax = sns.boxplot(
    data=metrics_errors_df,
    x="label",
    y="normalized_field_error/mean",
)
plt.show()

In [ ]:
metrics_errors_df.groupby("label")[
    [
        "normalized_field_error/mean",
        "normalized_coil_lengths/mean",
        "normalized_coil_to_coil_min_distances/mean",
        "normalized_coil_to_plasma_min_distances/mean",
        "normalized_coil_curvatures/mean",
    ]
].agg(["mean", "std"]).T